# P1 데이터 정제 흐름

`cleaning.py`와 `validate.py`를 사용해 원본 진단 → 컬럼명 통일 → 중복 제거 → 날짜·등급 정규화 → 센티널 처리 → 결측 대체 → 품질 검증 → 시각화 순서로 진행합니다.

## 1. 환경과 원본 데이터

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'module1_preprocessing/cleaning.py').exists()), None)
if ROOT is None:
    candidate = cwd / 'Metal_Recovery_Optimizer'
    if (candidate / 'module1_preprocessing/cleaning.py').exists():
        ROOT = candidate
if ROOT is None:
    raise FileNotFoundError('Metal_Recovery_Optimizer 또는 module1_preprocessing/notebooks 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from module1_preprocessing import cleaning as p1
from module1_preprocessing import validate as quality

# validate 모듈의 폰트 설정을 현재 환경에 맞게 덮어씁니다.
from matplotlib import font_manager
fonts = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic'] if f in fonts), 'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False

RAW_PATH = ROOT / 'data/raw' / '영풍그룹_금속회수율최적화_RawMaterialData_학생배포용.xlsx'
if not RAW_PATH.is_file():
    raise FileNotFoundError(
        f'원본 엑셀을 프로젝트의 data/raw/{RAW_PATH.name}에 넣고 다시 실행하세요.'
    )
raw = p1.load_raw(str(RAW_PATH))
dictionary = pd.read_excel(RAW_PATH, sheet_name=quality.DICT_SHEET)
print('원본:', RAW_PATH.relative_to(ROOT))
print('크기:', raw.shape)
display(raw.head(), dictionary)

history = []
def snapshot(name, frame):
    history.append({'단계': name, '행': len(frame), '열': frame.shape[1],
                    '결측 셀': int(frame.isna().sum().sum())})
snapshot('원본', raw)

## 2. 원본 품질 진단
결측 수, 자료형, 완전 동일 중복을 계산

In [ ]:
display(pd.DataFrame({'자료형': raw.dtypes.astype(str), '결측 수': raw.isna().sum()}))
print('완전 동일 중복:', int(raw.duplicated().sum()))
display(raw.loc[raw.duplicated(keep=False)].sort_values('배치ID'))

## 3. 컬럼명 통일과 중복 제거
한글 컬럼을 계약용 영문 이름으로 바꾸고, 모든 값이 동일한 중복만 제거합니다. 같은 ID에 다른 값이 있는 경우는 따로 확인해야 합니다.

In [ ]:
display(pd.DataFrame(p1.COLUMN_MAP.items(), columns=['원본 컬럼', '변환 컬럼']))
renamed = p1.normalize_columns(raw)
dedup = p1.drop_duplicate_rows(renamed)
snapshot('중복 제거', dedup)
print(f'{len(raw)}행 → {len(dedup)}행')
print('중복 제거 후 ID 중복:', int(dedup['batch_id'].duplicated().sum()))
assert dedup['batch_id'].notna().all() and dedup['batch_id'].is_unique
display(dedup.head())

## 4. 날짜와 원료등급 정규화
날짜는 P1의 명시적 6종 포맷으로 해석합니다. `03.01.2025`는 일·월·년, `01/03/25`는 월·일·년입니다.
등급은 공백, 소문자, `급` 접미사를 정리해 A/B/C로 통일합니다. 날짜 결측은 그대로 유지합니다.

In [ ]:
normalized = dedup.copy()
normalized['analysis_date'] = p1.parse_dates(dedup['analysis_date'])
normalized['grade'] = p1.normalize_grade(dedup['grade'])
display(pd.DataFrame({'원본 날짜': dedup['analysis_date'], '변환 날짜': normalized['analysis_date'],
                      '원본 등급': dedup['grade'], '변환 등급': normalized['grade']}).head(20))
parse_fail = dedup['analysis_date'].notna() & normalized['analysis_date'].isna()
print('원래 날짜 결측:', int(dedup['analysis_date'].isna().sum()))
print('값이 있었지만 변환 실패:', int(parse_fail.sum()))
display(dedup.loc[parse_fail, ['batch_id', 'analysis_date']])
display(normalized['grade'].value_counts().rename('행 수').to_frame())
snapshot('날짜·등급 정규화', normalized)

## 5. 센티널을 결측으로 변환
Ni 함량 999, 황산 15000, 온도 -5, 시간 0을 결측으로 바꿉니다.
행은 유지하고 그 외 데이터사전 범위 밖 경계값도 유지합니다.
처리 건수와 처리 전후 값을 확인합니다.

In [ ]:
sentinel_hits = pd.DataFrame({col: normalized[col].isin(values) for col, values in p1.SENTINELS.items()})
display(sentinel_hits.sum().rename('처리 셀 수').to_frame())
affected = sentinel_hits.any(axis=1)
masked = p1.mask_sentinels(normalized).loc[:, list(p1.COLUMN_MAP.values())].copy()
display(normalized.loc[affected, ['batch_id'] + list(p1.SENTINELS)])
display(masked.loc[affected, ['batch_id'] + list(p1.SENTINELS)])
snapshot('센티널 처리', masked)

## 6. 등급별 중앙값 대치
기존 P1 기본값인 `grade_median`을 실행합니다. 같은 등급의 중앙값을 사용하고, 해당 등급에서 계산할 수 없으면 전체 중앙값으로 보정합니다.

**모델링 연결 시 검토:** 기존 `clean()`은 전달된 전체 데이터에서 중앙값을 계산합니다. 예측 성능 평가에서는 데이터를 먼저 분할하고 학습 데이터에서만 대치 기준을 계산해야 합니다. 아래 결과는 기존 P1 흐름을 재현한 정제 결과입니다.

In [ ]:
display(masked.groupby('grade', observed=True)[p1.FEATURE_COLS].median())
imputed = p1.impute(masked, strategy='grade_median').loc[:, list(p1.COLUMN_MAP.values())].copy()
display(pd.DataFrame({'대치 전 결측': masked[p1.FEATURE_COLS + p1.TARGET_COLS].isna().sum(),
                      '대치 후 결측': imputed[p1.FEATURE_COLS + p1.TARGET_COLS].isna().sum()}))
example_col = 'ni_pct'
missing = masked[example_col].isna()
display(pd.DataFrame({'배치': masked.loc[missing, 'batch_id'], '등급': masked.loc[missing, 'grade'],
                      '대치 전': masked.loc[missing, example_col], '대치 후': imputed.loc[missing, example_col]}).head(15))
snapshot('등급별 중앙값 대치', imputed)

## 7. 기존 파이프라인과 일치하는지 검증
단계별 결과를 기존 `clean()`의 컬럼 순서로 정렬하고 값·자료형까지 비교합니다. 식별자, 등급, 날짜 자료형, 모델 입력 결측과 유한값도 확인합니다.

In [ ]:
reference = p1.clean(raw).loc[:, list(p1.COLUMN_MAP.values())]
cleaned = imputed.loc[:, reference.columns].copy()
pd.testing.assert_frame_equal(cleaned, reference)
assert cleaned['batch_id'].is_unique
assert cleaned['grade'].isin(['A', 'B', 'C']).all()
assert pd.api.types.is_datetime64_any_dtype(cleaned['analysis_date'])
assert cleaned[p1.FEATURE_COLS + p1.TARGET_COLS].notna().all().all()
assert np.isfinite(cleaned[p1.FEATURE_COLS + p1.TARGET_COLS].to_numpy(dtype=float)).all()
assert cleaned[p1.TARGET_COLS].ge(0).all().all() and cleaned[p1.TARGET_COLS].le(100).all().all()
print('단계별 결과와 clean(raw)가 동일합니다. 기본 정합성 검증 통과.')
display(pd.DataFrame(history))

## 8. 데이터사전 범위 위반 확인
센티널과 경계값을 구분합니다. 기존 P1은 범위 밖 경계값을 유지하고 보고하므로, 아래 위반 목록은 실패 판정이 아니라 추가 검토 대상입니다. 대치된 값이 범위를 벗어나는지도 함께 확인합니다.

In [ ]:
ranges = quality.load_valid_ranges(str(RAW_PATH))
range_summary, range_detail = quality.range_violation_report(cleaned, ranges)
display(range_summary, range_detail)
print('범위 위반 셀 수:', int(range_summary['위반 건수'].sum()))

## 9. 처리 전후 시각화
중복 제거 후 동일한 행 집합으로 비교합니다. 센티널을 제외한 관측값과 대치 후 값을 비교해 큰 센티널에 의해 축이 왜곡되는 것을 피합니다. 분포 변화만으로 대치 방법의 예측 성능을 판단할 수는 없습니다.

In [ ]:
from io import BytesIO
from IPython.display import Image

def show_figure(figure):
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))

missing_before = masked[p1.FEATURE_COLS].isna()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(missing_before.to_numpy(), aspect='auto', cmap='gray_r', interpolation='nearest')
axes[0].set_xticks(range(len(p1.FEATURE_COLS)), p1.FEATURE_COLS, rotation=90)
axes[0].set_title('대치 전 결측 위치 (검정 = 결측)')
axes[0].set_ylabel('중복 제거 후 행 순서')
pd.DataFrame({'원본 결측': dedup[p1.FEATURE_COLS].isna().sum(),
              '센티널 처리 후': masked[p1.FEATURE_COLS].isna().sum(),
              '대채 후': cleaned[p1.FEATURE_COLS].isna().sum()}).plot.bar(ax=axes[1])
axes[1].set_ylabel('결측 셀 수')
fig.tight_layout()
show_figure(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['ni_pct', 'time_hr', 'temp_c']):
    before = masked[col].dropna().to_numpy(dtype=float)
    after = cleaned[col].to_numpy(dtype=float)
    bins = np.histogram_bin_edges(np.concatenate([before, after]), bins=20)
    ax.hist(before, bins=bins, alpha=.5, label='센티널 제외 관측값')
    ax.hist(after, bins=bins, alpha=.5, label='대채 후')
    ax.set_title(col)
    ax.legend()
fig.tight_layout()
show_figure(fig)
plt.close(fig)

## 10. P2로 전달 및 결과 저장
P2는 `p1_proccessed.csv`의 처리값으로 EDA·파생변수 생성·검증을 수행합니다.
날짜 결측은 시간순 평가에서 보류합니다.

`SAVE_RESULTS=True`로 설정하면 `data/processed/p1_flow_notebook/`에 저장합니다.
- `p1_proccessed.csv`: 기본 16열 정제 데이터
- `range_summary.csv`: 정상 범위 위반 요약

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    output_dir = ROOT / 'data/processed/p1_flow_notebook'
    output_dir.mkdir(parents=True, exist_ok=True)
    data_cols = list(p1.COLUMN_MAP.values())
    cleaned[data_cols].to_csv(output_dir / 'p1_proccessed.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    range_summary.to_csv(output_dir / 'range_summary.csv', index=False, encoding='utf-8-sig')
    print('저장 위치:', output_dir.relative_to(ROOT))
else:
    print('저장하지 않았습니다. cleaned 변수에서 최종 정제본을 확인하세요.')